In [ ]:
#!/usr/bin/env python3
"""
Word-sense disambiguation and selectional-feature tagging for UD English EWT.

Compatible with Python 3.9.21.

Install dependencies:

    python -m pip install conllu nltk openai

The first run downloads the NLTK WordNet resources if they are absent.
"""

from __future__ import annotations

import hashlib
import json
import os
import re
import time
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Sequence, Set, Tuple

import nltk
from conllu import parse_incr
from nltk.corpus import wordnet as wn
from nltk.stem import WordNetLemmatizer
from openai import OpenAI


# =============================================================================
# Configuration
# =============================================================================

CONLLU_PATH = Path(
    "UD_English-EWT-master/"
    "UD_English-EWT-master/"
    "en_ewt-ud-train.conllu"
)

LM_STUDIO_BASE_URL = "http://172.26.176.1:1234/v1"
LM_STUDIO_API_KEY = "lm-studio"
#LM_STUDIO_MODEL = "qwen3.6-27b-mtp@q2_k_xl"
#LM_STUDIO_MODEL = "microsoft/phi-4-reasoning-plus"
#LM_STUDIO_MODEL = "qwen3.6-35b-a3b-mtp"
#LM_STUDIO_MODEL = "qwen3.8-27b"
LM_STUDIO_MODEL = "qwen3.8-27b@q2_k_xl"
#LM_STUDIO_MODEL = "qwen3.8-27b@iq3_xxs"



OUTPUT_PATH = Path("ewt_wordnet_wsd_features.jsonl")
CACHE_PATH = Path("ewt_wordnet_wsd_llm_cache.json")

# Number of neighbouring sentences available as document context.
CONTEXT_SENTENCES_EACH_SIDE = 2

# Maximum number of earlier WSD decisions included in each prompt.
MAX_PRIOR_WSD_DECISIONS = 20

# Set to an integer for testing, or None for the complete corpus.
MAX_SENTENCES: Optional[int] = 7000

MAX_LLM_RETRIES = 10
RETRY_WAIT_SECONDS = 2.0

CACHE_SAVE_INTERVAL = 20

# Separate deterministic settings for the two kinds of request.
WSD_TEMPERATURE = 0.6
FEATURE_TEMPERATURE = 0.6

# The WSD answer should contain only a numbered choice.
WSD_MAX_TOKENS = 10_000

# Feature JSON requires a larger allowance.
FEATURE_MAX_TOKENS = 10_000


# =============================================================================
# NLTK initialisation
# =============================================================================

def ensure_nltk_resources() -> None:
    """Download WordNet data only when it is unavailable."""

    resources = {
        "wordnet": "corpora/wordnet",
        "omw-1.4": "corpora/omw-1.4",
    }

    for package, resource_path in resources.items():
        try:
            nltk.data.find(resource_path)
        except LookupError:
            nltk.download(package, quiet=False)


ensure_nltk_resources()

LEMMATIZER = WordNetLemmatizer()


# =============================================================================
# LM Studio client
# =============================================================================

CLIENT = OpenAI(
    base_url=LM_STUDIO_BASE_URL,
    api_key=LM_STUDIO_API_KEY,
    timeout=680.0,
)


# =============================================================================
# Feature schema
# =============================================================================

WORDNET_FEATURE_TRIGGERS: Dict[str, Set[str]] = {
    # "Root": {
    #     "entity.n.01",
    # },
    "Concrete": {
        "physical_entity.n.01",
    },
    "Organisation": {
        "organization.n.01",
        "social_group.n.01",
    },
    "Natural_Force": {
        "natural_phenomenon.n.01",
    },
    "Animate": {
        "living_thing.n.01",
        "organism.n.01",
    },
    "Body_Part": {
        "body_part.n.01",
    },
    "Human": {
        "person.n.01",
        "people.n.01",
    },
    "Animal": {
        "animal.n.01",
    },
    "Plant": {
        "plant.n.02",
    },
    "Machine": {
        "instrumentality.n.03",
        "machine.n.01",
    },
    "Vehicle": {
        "vehicle.n.01",
        "public_transport.n.01",
    },
    "Natural": {
        "natural_phenomenon.n.01",
        "natural_object.n.01",
    },
    "Artefact": {
        "artifact.n.01",
    },
    "Garment": {
        "clothing.n.01",
    },
    "Comestible": {
        "food.n.01",
        "food.n.02",
    },
    "Substance": {
        "substance.n.01",
    },
    "Fluid": {
        "fluid.n.01",
    },
    "Abstract": {
        "abstraction.n.06",
    },
    "Language": {
        "natural_language.n.01",
    },
    "Sound": {
        "sound.n.01",
        "sound.n.02",
        "sound.n.04",
        "auditory_communication.n.01",
    },
    "Communication": {
        "communication.n.02",
    },
    "Eventive": {
        "event.n.01",
    },
    "Currency": {
        "monetary_unit.n.01",
        "medium_of_exchange.n.01"
    },
    "Location": {
        "location.n.01",
    },
}


FEATURE_PARENTS: Dict[str, Set[str]] = {
    # -------------------------------------------------------------------------
    # Root branches
    # -------------------------------------------------------------------------

    # "Internal_Control": {
    #     "Root",
    # },
    # "Concrete": {
    #     "Root",
    # },
    # "Time": {
    #     "Root",
    # },
    # "State": {
    #     "Root",
    # },
    # "Abstract": {
    #     "Root",
    # },
    # "Currency": {
    #     "Root",
    # },
    # "Location": {
    #     "Root",
    # },

    # -------------------------------------------------------------------------
    # Internal Control branch
    # -------------------------------------------------------------------------

    "Organisation": {
        "Internal_Control",
        "Abstract",          # organisations are also abstract entities
    },
    "Natural_Force": {
        "Internal_Control",
    },
    "Animate": {
        "Internal_Control",
    },
    "Human": {
        "Animate",
    },
    "Feminine": {
        "Human",
    },
    "Masculine": {
        "Human",
    },
    "Unspecified_Gender_or_Neuter": {
        "Human",
    },
    "Animal": {
        "Animate",
    },
    "Plant": {
        "Animate",
    },
    "Body_Part": {
        "Animate",
    },
    "Machine": {
        "Internal_Control",
        "Artefact",
    },
    "Vehicle": {
        "Machine",
    },

    # -------------------------------------------------------------------------
    # Concrete branch
    # -------------------------------------------------------------------------

    "Natural": {
        "Concrete",
    },
    "Artefact": {
        "Concrete",
    },
    "Comestible": {
        "Concrete",
    },
    "Substance": {
        "Concrete",
    },
    "State_of_Matter": {
        "Concrete",
    },

    # Concrete
    "Inanimate": {
        "Concrete",
    },
    "Animate": {
        "Natural",
    },

    # Artefact
    "Garment": {
        "Artefact",
    },
    "Machine": {
        "Artefact",
        "Internal_Control",
    },
    "Vehicle": {
        "Machine",
    },

    # State of Matter
    "Solid": {
        "State_of_Matter",
    },
    "Fluid": {
        "State_of_Matter",
    },

    # Solid
    "Rigidity": {
        "Solid",
    },
    "Shape": {
        "Solid",
    },

    # Rigidity
    "Rigid": {
        "Rigidity",
    },
    "Non_Rigid": {
        "Rigidity",
    },

    # Shape
    "Elongated": {
        "Shape",
    },
    "Pointy": {
        "Shape",
    },

    # -------------------------------------------------------------------------
    # Time
    # -------------------------------------------------------------------------

    "Point_in_Time": {
        "Time",
    },
    "Duration": {
        "Time",
    },

    # -------------------------------------------------------------------------
    # Abstract
    # -------------------------------------------------------------------------

    "Language": {
        "Abstract",
    },
    "Sound": {
        "Abstract",
    },
    "Communication": {
        "Abstract",
    },
    "Eventive": {
        "Abstract",
    },
    "Organisation": {
        "Internal_Control",
        "Abstract",
    },

    # Communication
    "Statement": {
        "Communication",
    },
    "Question": {
        "Communication",
    },
}


LLM_DERIVED_FEATURES: Tuple[str, ...] = (
    "Solid",
    "Rigid",
    "Non_Rigid",
    "Elongated",
    "Pointy",
    "Time",
    "Point_in_Time",
    "Duration",
    "State",
    "Communication",
    "Statement",
    "Question",
    "Masculine",
    "Feminine",
    "Unspecified_Gender_or_Neuter",
    "Internal_Control",
    "Inanimate",
    "Location",
    "Organisation",
    "Resource",
)


PROPER_NOUN_FEATURES: Tuple[str, ...] = (
    "Organisation",
    "Human",
    "Masculine",
    "Feminine",
    "Unspecified_Gender_or_Neuter",
    "Animal",
    "Vehicle",
    "Language",
    "Eventive",
    "Location",
    "Web_Location",
    "Time",
    "Point_in_Time",
    "Duration",
)


# =============================================================================
# General file and cache helpers
# =============================================================================

def load_json_file(path: Path, default: Any) -> Any:
    """Load JSON, returning a default value when no valid file exists."""

    if not path.exists():
        return default

    try:
        with path.open("r", encoding="utf-8") as handle:
            return json.load(handle)
    except (OSError, json.JSONDecodeError):
        return default


def atomic_write_json(path: Path, data: Any) -> None:
    """Write JSON safely by replacing the destination atomically."""

    temporary_path = path.with_suffix(path.suffix + ".tmp")

    with temporary_path.open("w", encoding="utf-8") as handle:
        json.dump(
            data,
            handle,
            ensure_ascii=False,
            indent=2,
            sort_keys=True,
        )

    temporary_path.replace(path)


LLM_CACHE: Dict[str, Any] = load_json_file(CACHE_PATH, {})
NEW_CACHE_ENTRIES = 0


def make_cache_key(task: str, payload: Dict[str, Any]) -> str:
    """Create a stable cache key from a task and its complete input."""

    serialised = json.dumps(
        {
            "task": task,
            "model": LM_STUDIO_MODEL,
            "payload": payload,
        },
        ensure_ascii=False,
        sort_keys=True,
    )

    digest = hashlib.sha256(serialised.encode("utf-8")).hexdigest()
    return f"{task}:{digest}"


def save_llm_cache(force: bool = False) -> None:
    """Save the cache periodically or when explicitly forced."""

    global NEW_CACHE_ENTRIES

    if force or NEW_CACHE_ENTRIES >= CACHE_SAVE_INTERVAL:
        atomic_write_json(CACHE_PATH, LLM_CACHE)
        NEW_CACHE_ENTRIES = 0


def cache_result(cache_key: str, result: Dict[str, Any]) -> None:
    """Add one result to the in-memory cache."""

    global NEW_CACHE_ENTRIES

    LLM_CACHE[cache_key] = result
    NEW_CACHE_ENTRIES += 1
    save_llm_cache()


# =============================================================================
# LM response parsing
# =============================================================================

def get_message_text(response: Any) -> str:
    """
    Recover final content from an OpenAI-compatible response.

    Some reasoning models expose content in `content`, while some servers may
    expose additional text through `reasoning_content`.
    """

    message = response.choices[0].message

    content = getattr(message, "content", None)
    if content:
        return str(content).strip()

    reasoning_content = getattr(message, "reasoning_content", None)
    if reasoning_content:
        return str(reasoning_content).strip()

    return ""


def extract_json_object(text: str) -> Dict[str, Any]:
    """
    Extract a JSON object from plain text, a Markdown fence, or surrounding
    incidental text.
    """

    text = text.strip()

    fence_match = re.search(
        r"```(?:json)?\s*(\{.*?\})\s*```",
        text,
        flags=re.DOTALL | re.IGNORECASE,
    )

    if fence_match:
        text = fence_match.group(1)

    try:
        value = json.loads(text)

        if not isinstance(value, dict):
            raise ValueError("Expected a JSON object.")

        return value

    except json.JSONDecodeError:
        pass

    first_brace = text.find("{")
    last_brace = text.rfind("}")

    if first_brace == -1 or last_brace <= first_brace:
        raise ValueError(
            f"No JSON object found in response: {text!r}"
        )

    value = json.loads(text[first_brace:last_brace + 1])

    if not isinstance(value, dict):
        raise ValueError("Expected a JSON object.")

    return value


FINAL_CHOICE_PATTERN = re.compile(
    r"FINAL_CHOICE\s*:\s*(\d+)",
    flags=re.IGNORECASE,
)


def extract_final_choice(text: str, option_count: int) -> int:
    """
    Extract `FINAL_CHOICE: n` from a response.

    The `<end_generation>` suffix is intentionally optional because LM Studio
    may remove the matched stop sequence from returned content.
    """

    match = FINAL_CHOICE_PATTERN.search(text)

    if not match:
        raise ValueError(
            "The WSD response did not contain 'FINAL_CHOICE: [number]'. "
            f"Raw response: {text!r}"
        )

    choice = int(match.group(1))

    if not 0 <= choice <= option_count:
        raise ValueError(
            f"WSD choice {choice} is outside the valid range "
            f"0–{option_count}."
        )

    return choice


# =============================================================================
# LM Studio calls
# =============================================================================

def call_llm_json(
    task: str,
    system_prompt: str,
    user_payload: Dict[str, Any],
) -> Dict[str, Any]:
    """Call LM Studio for one JSON feature-classification result."""

    cache_key = make_cache_key(task, user_payload)

    cached = LLM_CACHE.get(cache_key)
    if isinstance(cached, dict):
        return cached

    last_error: Optional[Exception] = None

    for attempt in range(1, MAX_LLM_RETRIES + 1):
        try:
            response = CLIENT.chat.completions.create(
                model=LM_STUDIO_MODEL,
                temperature=FEATURE_TEMPERATURE,
                max_tokens=FEATURE_MAX_TOKENS,
                messages=[
                    {
                        "role": "system",
                        "content": system_prompt,
                    },
                    {
                        "role": "user",
                        "content": json.dumps(
                            user_payload,
                            ensure_ascii=False,
                            indent=2,
                        ),
                    },
                ],
            )

            content = get_message_text(response)
            result = extract_json_object(content)

            cache_result(cache_key, result)
            return result

        except Exception as exc:
            last_error = exc

            if attempt < MAX_LLM_RETRIES:
                wait = RETRY_WAIT_SECONDS * attempt

                print(
                    f"Feature request failed on attempt {attempt}: {exc}. "
                    f"Retrying in {wait:.1f} seconds."
                )

                time.sleep(wait)

    raise RuntimeError(
        f"LM Studio feature request failed after "
        f"{MAX_LLM_RETRIES} attempts."
    ) from last_error


def call_llm_wsd_choice(
    prompt: str,
    cache_payload: Dict[str, Any],
    option_count: int,
) -> Dict[str, Any]:
    """
    Call LM Studio for a compact numbered WSD decision.

    This avoids JSON generation and allows only a very small response.
    """

    cache_key = make_cache_key(
        task="wordnet_wsd_numbered_choice_v2",
        payload=cache_payload,
    )

    cached = LLM_CACHE.get(cache_key)
    if isinstance(cached, dict):
        return cached

    last_error: Optional[Exception] = None

    for attempt in range(1, MAX_LLM_RETRIES + 1):
        try:
            retry_instruction = ""

            if attempt > 1:
                retry_instruction = (
                    "\n\nYour previous response did not match the required "
                    "format. Return only FINAL_CHOICE: [number] "
                    "<end_generation>."
                )

            response = CLIENT.chat.completions.create(
                model=LM_STUDIO_MODEL,
                temperature=WSD_TEMPERATURE,
                max_tokens=WSD_MAX_TOKENS,
                stop=["<end_generation>"],
                messages=[
                    {
                        "role": "user",
                        "content": prompt + retry_instruction,
                    }
                ],
            )

            raw_response = get_message_text(response)
            choice = extract_final_choice(
                text=raw_response,
                option_count=option_count,
            )

            result = {
                "choice": choice,
                "raw_response": raw_response,
                "method": "llm_numbered_choice",
            }

            cache_result(cache_key, result)
            return result

        except Exception as exc:
            last_error = exc

            if attempt < MAX_LLM_RETRIES:
                wait = RETRY_WAIT_SECONDS * attempt

                print(
                    f"WSD request failed on attempt {attempt}: {exc}. "
                    f"Retrying in {wait:.1f} seconds."
                )

                time.sleep(wait)

    raise RuntimeError(
        f"LM Studio WSD request failed after "
        f"{MAX_LLM_RETRIES} attempts."
    ) from last_error


def normalise_boolean(value: Any) -> bool:
    """Convert conservative JSON-like truth values to bool."""

    if isinstance(value, bool):
        return value

    if isinstance(value, int):
        return value == 1

    if isinstance(value, str):
        return value.strip().lower() in {
            "true",
            "yes",
            "1",
        }

    return False


# =============================================================================
# CoNLL-U loading
# =============================================================================

def load_conllu(path: Path) -> List[Any]:
    """Load a CoNLL-U file as a list of TokenList objects."""

    if not path.exists():
        raise FileNotFoundError(
            f"Could not find the CoNLL-U file:\n{path.resolve()}"
        )

    sentences: List[Any] = []

    with path.open("r", encoding="utf-8") as handle:
        for sentence_number, sentence in enumerate(parse_incr(handle)):
            sentences.append(sentence)

            if (
                MAX_SENTENCES is not None
                and sentence_number + 1 >= MAX_SENTENCES
            ):
                break

    return sentences


def sentence_text(sentence: Any) -> str:
    """Return metadata text, reconstructing the sentence when absent."""

    metadata_text = sentence.metadata.get("text")

    if metadata_text:
        return str(metadata_text)

    forms: List[str] = []

    for token in sentence:
        token_id = token.get("id")

        if not isinstance(token_id, int):
            continue

        form = token.get("form")

        if form:
            forms.append(str(form))

    return " ".join(forms)


def build_context(
    sentences: Sequence[Any],
    sentence_index: int,
    radius: int = CONTEXT_SENTENCES_EACH_SIDE,
) -> Dict[str, Any]:
    """Build neighbouring discourse context."""

    previous_start = max(0, sentence_index - radius)
    following_end = min(
        len(sentences),
        sentence_index + radius + 1,
    )

    return {
        "previous_sentences": [
            sentence_text(sentences[index])
            for index in range(previous_start, sentence_index)
        ],
        "sentence": sentence_text(sentences[sentence_index]),
        "following_sentences": [
            sentence_text(sentences[index])
            for index in range(
                sentence_index + 1,
                following_end,
            )
        ],
    }


# =============================================================================
# WordNet candidate senses
# =============================================================================

def wordnet_pos_for_upos(upos: str) -> Optional[str]:
    """Map Universal Dependencies POS to WordNet POS."""

    return {
        "NOUN": wn.NOUN,
        #"VERB": wn.VERB,
    }.get(upos)


def get_wordnet_candidates(
    form: str,
    lemma: Optional[str],
    upos: str,
) -> List[Any]:
    """Find WordNet candidate synsets for a noun or verb."""

    wn_pos = wordnet_pos_for_upos(upos)

    if wn_pos is None:
        return []

    search_forms: List[str] = []

    for candidate_form in (
        lemma,
        form,
        LEMMATIZER.lemmatize(
            (lemma or form).lower(),
            pos=wn_pos,
        ),
        LEMMATIZER.lemmatize(
            form.lower(),
            pos=wn_pos,
        ),
    ):
        if not candidate_form:
            continue

        normalised = (
            str(candidate_form)
            .strip()
            .lower()
            .replace(" ", "_")
        )

        if normalised and normalised not in search_forms:
            search_forms.append(normalised)

    synsets: List[Any] = []
    seen: Set[str] = set()

    for search_form in search_forms:
        for synset in wn.synsets(search_form, pos=wn_pos):
            if synset.name() in seen:
                continue

            synsets.append(synset)
            seen.add(synset.name())

    return synsets


def synset_as_candidate(synset: Any) -> Dict[str, Any]:
    """Convert an NLTK synset into a compact serialisable record."""

    return {
        "synset": synset.name(),
        "lemmas": [
            lemma.name().replace("_", " ")
            for lemma in synset.lemmas()
        ],
        "definition": synset.definition(),
        "examples": list(synset.examples()),
        "lexname": synset.lexname(),
    }


# =============================================================================
# Dependency context
# =============================================================================

def token_dependency_context(
    sentence: Any,
    target_token: Any,
) -> Dict[str, Any]:
    """Describe the target token's immediate UD environment."""

    target_id = target_token.get("id")
    head_id = target_token.get("head")

    head_record: Optional[Dict[str, Any]] = None
    dependants: List[Dict[str, Any]] = []

    for token in sentence:
        token_id = token.get("id")

        if not isinstance(token_id, int):
            continue

        if token_id == head_id:
            head_record = {
                "id": token_id,
                "form": token.get("form"),
                "lemma": token.get("lemma"),
                "upos": token.get("upos"),
                "deprel": token.get("deprel"),
            }

        if token.get("head") == target_id:
            dependants.append(
                {
                    "id": token_id,
                    "form": token.get("form"),
                    "lemma": token.get("lemma"),
                    "upos": token.get("upos"),
                    "deprel": token.get("deprel"),
                }
            )

    return {
        "dependency_relation": target_token.get("deprel"),
        "head": head_record,
        "dependants": dependants,
    }


def format_dependency_context(
    dependency_context: Dict[str, Any],
) -> str:
    """Format dependency information compactly for the WSD prompt."""

    lines: List[str] = []

    relation = dependency_context.get("dependency_relation")
    lines.append(f"Target dependency relation: {relation or 'unknown'}")

    head = dependency_context.get("head")

    if isinstance(head, dict):
        lines.append(
            "Head: "
            f"{head.get('form')} "
            f"[lemma={head.get('lemma')}; "
            f"POS={head.get('upos')}; "
            f"relation={head.get('deprel')}]"
        )
    else:
        lines.append("Head: ROOT or unavailable")

    dependants = dependency_context.get("dependants", [])

    if dependants:
        dependant_strings = []

        for dependant in dependants:
            dependant_strings.append(
                f"{dependant.get('form')} "
                f"[{dependant.get('upos')}; "
                f"{dependant.get('deprel')}]"
            )

        lines.append(
            "Dependants: " + "; ".join(dependant_strings)
        )
    else:
        lines.append("Dependants: none")

    return "\n".join(lines)


# =============================================================================
# Hypernym paths
# =============================================================================

def get_noun_hypernym_paths(synset: Any) -> List[List[Any]]:
    """
    Return all noun hypernym paths ordered from entity.n.01 to the selected
    sense.
    """

    if synset.pos() != wn.NOUN:
        return []

    paths: List[List[Any]] = []

    for path in synset.hypernym_paths():
        ordered = list(path)

        if ordered and ordered[0].name() != "entity.n.01":
            ordered.reverse()

        paths.append(ordered)

    unique_paths: List[List[Any]] = []
    seen: Set[Tuple[str, ...]] = set()

    for path in paths:
        key = tuple(item.name() for item in path)

        if key not in seen:
            unique_paths.append(path)
            seen.add(key)

    return unique_paths


def serialise_hypernym_paths(
    paths: Sequence[Sequence[Any]],
) -> List[List[Dict[str, str]]]:
    """Convert hypernym paths into JSON-compatible records."""

    return [
        [
            {
                "synset": synset.name(),
                "definition": synset.definition(),
            }
            for synset in path
        ]
        for path in paths
    ]


def compact_candidate_features(synset: Any) -> List[str]:
    """
    Derive compact ontology features for a noun candidate.

    These are included as supporting evidence in the WSD options. They are not
    treated as selected-token features until that sense has actually been
    chosen.
    """

    if synset.pos() != wn.NOUN:
        return []

    paths = get_noun_hypernym_paths(synset)
    features, _ = features_from_hypernym_paths(paths)

    return sorted(features)


def compact_hypernym_trail(synset: Any) -> str:
    """Return the shortest readable noun hypernym trail."""

    if synset.pos() != wn.NOUN:
        return ""

    paths = get_noun_hypernym_paths(synset)

    if not paths:
        return ""

    shortest_path = min(paths, key=len)

    return " > ".join(
        item.name()
        for item in shortest_path
    )


# =============================================================================
# Earlier WSD history
# =============================================================================

WSDHistory = Dict[int, List[Dict[str, Any]]]


def add_wsd_history_entry(
    history_by_sentence: WSDHistory,
    sentence_index: int,
    token_id: int,
    form: str,
    lemma: Optional[str],
    upos: str,
    synset: str,
    definition: str,
) -> None:
    """Add or replace one successful decision in sentence history."""

    entries = history_by_sentence.setdefault(sentence_index, [])

    new_entry = {
        "token_id": token_id,
        "form": form,
        "lemma": lemma,
        "upos": upos,
        "synset": synset,
        "definition": definition,
    }

    for index, entry in enumerate(entries):
        if entry.get("token_id") == token_id:
            entries[index] = new_entry
            break
    else:
        entries.append(new_entry)

    entries.sort(
        key=lambda item: int(item.get("token_id", 0))
    )


def load_processing_state(
    path: Path,
) -> Tuple[Set[Tuple[int, str]], WSDHistory]:
    """
    Recover completed records and earlier successful WSD decisions from JSONL.
    """

    completed: Set[Tuple[int, str]] = set()
    history_by_sentence: WSDHistory = {}

    if not path.exists():
        return completed, history_by_sentence

    with path.open("r", encoding="utf-8") as handle:
        for line in handle:
            line = line.strip()

            if not line:
                continue

            try:
                record = json.loads(line)
            except json.JSONDecodeError:
                continue

            sentence_index = record.get("sentence_index")
            token_id = record.get("token_id")

            if not isinstance(sentence_index, int):
                continue

            if token_id is None:
                continue

            completed.add(
                (
                    sentence_index,
                    str(token_id),
                )
            )

            upos = record.get("upos")
            selected_synset = record.get("selected_synset")
            selected_definition = record.get("selected_definition")

            if (
                upos in {"NOUN", "VERB"}
                and selected_synset
                and isinstance(token_id, int)
            ):
                add_wsd_history_entry(
                    history_by_sentence=history_by_sentence,
                    sentence_index=sentence_index,
                    token_id=token_id,
                    form=str(record.get("form") or ""),
                    lemma=record.get("lemma"),
                    upos=str(upos),
                    synset=str(selected_synset),
                    definition=str(selected_definition or ""),
                )

    return completed, history_by_sentence


def get_prior_wsd_entries(
    history_by_sentence: WSDHistory,
    sentence_index: int,
    current_token_id: int,
) -> List[Dict[str, Any]]:
    """Return only decisions preceding the current token."""

    entries = history_by_sentence.get(sentence_index, [])

    prior_entries = [
        entry
        for entry in entries
        if int(entry.get("token_id", 0)) < current_token_id
    ]

    return prior_entries[-MAX_PRIOR_WSD_DECISIONS:]


def format_prior_wsd_history(
    entries: Sequence[Dict[str, Any]],
) -> str:
    """Format earlier decisions for insertion into the prompt."""

    if not entries:
        return "None."

    lines: List[str] = []

    for entry in entries:
        lines.append(
            f"- Position {entry.get('token_id')}: "
            f"{entry.get('form')!r} "
            f"({entry.get('upos')}) "
            f"→ {entry.get('synset')}: "
            f"{entry.get('definition')}"
        )

    return "\n".join(lines)


# =============================================================================
# Fast numbered-choice WSD
# =============================================================================

def format_wsd_options(
    candidates: Sequence[Any],
) -> str:
    """Format WordNet candidates as numbered WSD options."""

    option_blocks: List[str] = []

    for option_number, synset in enumerate(candidates, start=1):
        lemmas = ", ".join(
            lemma.name().replace("_", " ")
            for lemma in synset.lemmas()
        )

        examples = list(synset.examples())

        if examples:
            examples_text = " | ".join(examples[:3])
        else:
            examples_text = "None supplied by WordNet."

        lines = [
            f"{option_number}. {synset.name()}",
            f"   Definition: {synset.definition()}",
            f"   Lemmas: {lemmas}",
            f"   Lexical category: {synset.lexname()}",
            f"   Examples: {examples_text}",
        ]

        if synset.pos() == wn.NOUN:
            feature_text = ", ".join(
                compact_candidate_features(synset)
            )

            hypernym_text = compact_hypernym_trail(synset)

            lines.append(
                "   Ontology features: "
                f"{feature_text or 'None identified'}"
            )
            lines.append(
                "   Hypernym trail: "
                f"{hypernym_text or 'Unavailable'}"
            )

        option_blocks.append("\n".join(lines))

    return "\n\n".join(option_blocks)


def build_wsd_prompt(
    sentence: Any,
    sentence_index: int,
    sentences: Sequence[Any],
    token: Any,
    candidates: Sequence[Any],
    prior_wsd_entries: Sequence[Dict[str, Any]],
) -> str:
    """Construct the compact numbered-choice WSD prompt."""

    token_text = str(token.get("form") or "")
    token_id = int(token.get("id"))
    token_position = token_id
    current_sentence = sentence_text(sentence)

    context = build_context(
        sentences=sentences,
        sentence_index=sentence_index,
    )

    previous_sentences = context["previous_sentences"]
    following_sentences = context["following_sentences"]

    previous_text = (
        "\n".join(f"- {text}" for text in previous_sentences)
        if previous_sentences
        else "None."
    )

    following_text = (
        "\n".join(f"- {text}" for text in following_sentences)
        if following_sentences
        else "None."
    )

    prior_wsd_history = format_prior_wsd_history(
        prior_wsd_entries
    )

    syntax_text = format_dependency_context(
        token_dependency_context(sentence, token)
    )

    all_options_str = format_wsd_options(candidates)

    sense_prompt = f"""
Reply only with:

FINAL_CHOICE: [number] <end_generation>

Replace [number] with the number of the chosen option.

If no option accurately expresses the meaning of the target token in this
sentence, output:

FINAL_CHOICE: 0 <end_generation>

Do not include explanations, analysis, Markdown, punctuation after the
terminator, or any other text.

TARGET TOKEN
Token: {token_text!r}
Lemma: {token.get("lemma")!r}
Part of speech: {token.get("upos")}
Position: {token_position}

CURRENT SENTENCE
{current_sentence}

IMMEDIATE SYNTACTIC CONTEXT
{syntax_text}

PREVIOUS SENTENCES
{previous_text}

FOLLOWING SENTENCES
{following_text}

EARLIER WSD DECISIONS IN THE CURRENT SENTENCE
{prior_wsd_history}

INSTRUCTIONS

1. Select the option whose definition exactly matches the meaning of
   {token_text!r} in the current sentence.
2. Consider the complete sentence, dependency context, WordNet definition,
   examples, lexical category, ontology features, and hypernym trail.
3. Be careful with metaphorical, metonymic, institutional, idiomatic, and
   eventive uses.
4. Treat earlier WSD decisions as established semantic context. Prefer a
   reading coherent with them, but do not copy an earlier sense merely because
   its wording is similar.
5. The current token must independently match the chosen definition.
6. Choose 0 when no supplied sense adequately matches.
7. Return only the required FINAL_CHOICE line.

POSSIBLE SENSE OPTIONS

{all_options_str}

Reply only with:

FINAL_CHOICE: [number] <end_generation>

Replace [number] with the number of the chosen option.

If no option accurately expresses the meaning of the target token in this
sentence, output:

FINAL_CHOICE: 0 <end_generation>

Do not include explanations, analysis, Markdown, punctuation after the
terminator, or any other text.

FINAL_CHOICE: [number]
""".strip()

    return sense_prompt


def disambiguate_token(
    sentence: Any,
    sentence_index: int,
    sentences: Sequence[Any],
    token: Any,
    candidates: Sequence[Any],
    prior_wsd_entries: Sequence[Dict[str, Any]],
) -> Tuple[Optional[Any], Dict[str, Any]]:
    """
    Select one validated WordNet candidate using a numbered-choice response.
    """

    if not candidates:
        return None, {
            "choice": 0,
            "method": "no_candidates",
        }

    # A single WordNet candidate does not require an LLM call.
    if len(candidates) == 1:
        selected = candidates[0]

        return selected, {
            "choice": 1,
            "method": "single_candidate",
            "raw_response": None,
        }

    prompt = build_wsd_prompt(
        sentence=sentence,
        sentence_index=sentence_index,
        sentences=sentences,
        token=token,
        candidates=candidates,
        prior_wsd_entries=prior_wsd_entries,
    )

    cache_payload = {
        "sentence_index": sentence_index,
        "sentence": sentence_text(sentence),
        "target": {
            "form": token.get("form"),
            "lemma": token.get("lemma"),
            "upos": token.get("upos"),
            "token_id": token.get("id"),
        },
        "prior_wsd_entries": list(prior_wsd_entries),
        "candidate_synsets": [
            synset.name()
            for synset in candidates
        ],
        "prompt_version": 2,
    }

    response = call_llm_wsd_choice(
        prompt=prompt,
        cache_payload=cache_payload,
        option_count=len(candidates),
    )

    choice = int(response["choice"])

    if choice == 0:
        return None, response

    selected_index = choice - 1

    if not 0 <= selected_index < len(candidates):
        response["validation_error"] = (
            f"Choice {choice} did not map to a candidate."
        )
        return None, response

    return candidates[selected_index], response


# =============================================================================
# Ontology-derived features
# =============================================================================

def expand_feature_inheritance(
    features: Iterable[str],
) -> Set[str]:
    """Recursively add all parent features."""

    expanded = set(features)
    agenda = list(expanded)

    while agenda:
        feature = agenda.pop()

        for parent in FEATURE_PARENTS.get(feature, set()):
            if parent not in expanded:
                expanded.add(parent)
                agenda.append(parent)

    return expanded


def features_from_hypernym_paths(
    paths: Sequence[Sequence[Any]],
) -> Tuple[Set[str], Dict[str, List[str]]]:
    """Map every synset in every hypernym path onto schema features."""

    path_synset_names: Set[str] = {
        synset.name()
        for path in paths
        for synset in path
    }

    direct_features: Set[str] = set()
    sources: Dict[str, List[str]] = {}

    for feature, trigger_names in WORDNET_FEATURE_TRIGGERS.items():
        matches = sorted(
            path_synset_names.intersection(trigger_names)
        )

        if matches:
            direct_features.add(feature)
            sources[feature] = [
                f"wordnet:{synset_name}"
                for synset_name in matches
            ]

    expanded_features = expand_feature_inheritance(
        direct_features
    )

    for feature in expanded_features:
        if feature not in sources:
            sources[feature] = ["inherited"]

    return expanded_features, sources


# =============================================================================
# LLM-derived common-noun features
# =============================================================================

NOUN_FEATURE_SYSTEM_PROMPT = """
You are assigning semantic selectional-restriction features to one
disambiguated common noun.

The WordNet sense has already been selected. Classify the referent denoted by
that sense in the supplied sentence, not every possible referent of the word.

Feature definitions:

Internal_Control:
    The referent can characteristically initiate, regulate, or direct its own
    actions or behaviour. Humans, many animals, organisations acting through
    members, and autonomous machines may qualify. Natural forces such as rain and wind also qualify.
    Mere movement or causation is insufficient.

Solid:
    The physical referent is neither a liquid nor a gas in the relevant use.
    Ordinary bounded objects and masses of solid material qualify.

Rigid:
    The solid referent normally resists bending and substantially retains its
    shape under ordinary handling.

Non_Rigid:
    The solid referent normally bends, folds, stretches, or deforms under
    ordinary handling. Rigid and Non_Rigid must not both be true.

Elongated:
    The referent's typical shape is markedly longer in one dimension than in
    its other dimensions.

Pointy:
    The referent typically has a pointed tip, sharp apex, or tapering end.

Time:
    The referent is a temporal entity, interval, date, age, period, or temporal position. The referent should be able to logically answer questions beginning with "when" or "how long" or "for how long".
    E.g. "for four hours", "next week", "in a day", "tomorrow", "on the 6th of July", "in July", "in 2 hours", "at 4:00 pm", "in three minutes".

Point_in_Time:
    The temporal referent is conceptualised as a bounded point or date rather than an extended interval. The referent should be able to logically answer questions beginning with "when".
    E.g. "at 4:00 pm", "tomorrow", "yesterday", "in July", "in 1980", "at the weekend", "on the weekend".
    Point_in_Time and Duration should not both be true unless the sentence genuinely coerces both readings.

Duration:
    The temporal referent is conceptualised as an interval or amount of time. The referent should be able to logically answer questions beginning with "how long" or "for how long".
    E.g. "I ran for four hours", "I ran to the next city in a day", "I trained hard for all of December", "I trained hard throughout the week".
    Point_in_Time and Duration should not both be true unless the sentence genuinely coerces both readings.

State:
    The referent is a resulting state caused by the verb. For example, in "He mashed the fruit into a pulp", "pulp" is the resulting state. In "they smashed the plates to pieces", "pieces" is the resulting state.

Communication:
    The referent is communicative content, a communicative act, or a message.

Statement:
    The referent is declarative or assertive communicative content.

Question:
    The referent is interrogative communicative content.

Location:
    The referent is a place, region, site, spatial area, or spatial position.
    An ordinary movable physical object is not a Location merely because it
    occupies space.

Organisation:
    The referent is an organisation, social group, or polity (including governments). 
    This includes locations used metonymically to refer to organisations.

Resource:
    The referent is something that can be wasted, spent, or expended like time, money, effort, intelligence, good will, etc.

Masculine:
    A human that is masculine in gender. This feature should only apply if the word explicitly encodes gender (e.g. man, son, boy). It should not be inferred from context.

Feminine:
    A human that is feminine in gender. This feature should only apply if the word explicitly encodes gender (e.g. woman, daughter, girl). It should not be inferred from context.

Unspecified_Gender_or_Neuter:
    The referent is Human, but the word does not explicitly encode gender features (e.g. pilot, doctor, patient, employee).

Rules:

1. Use the selected WordNet definition and sentence context.
2. Return one result for every requested feature.
3. Be conservative: false means the feature is not supported.
4. Respect the stated incompatibilities.
5. Do not add features that were not requested.
6. Give only a short justification for each decision.
7. Rigid, Non_Rigid, Elongated, and Pointy can be true only when Solid is true.
8. Countries and nation-states may be both Location and Organisation.

Gender rules:
- Apply a gender feature only when Human is true.
- Exactly one of Masculine, Feminine, and Unspecified_or_Neuter must be true
  when Human is true.
- Do not infer gender solely from an unfamiliar name.

Return exactly:

{
  "features": {
    "FeatureName": {
      "applies": true,
      "reason": "brief justification"
    }
  }
}
""".strip()


def choose_llm_noun_features(
    existing_features: Set[str],
) -> List[str]:
    """Select unresolved features relevant to the selected noun sense."""

    requested: Set[str] = {
        "Location",
        "Organisation",
    }

    if "Internal_Control" not in existing_features:
        requested.add("Internal_Control")

    if (
        "Concrete" in existing_features
        and "Animate" not in existing_features
    ):
        requested.add("Inanimate")

    if "Human" in existing_features:
        requested.update(
            {
                "Masculine",
                "Feminine",
                "Unspecified_Gender_or_Neuter",
            }
        )

    if (
        "Concrete" in existing_features
        and "Human" not in existing_features
        and "Animal" not in existing_features
    ):
        requested.update(
            {
                "Solid",
                "Rigid",
                "Non_Rigid",
                "Elongated",
                "Pointy",
            }
        )

    if (
        "Abstract" in existing_features
        or "Time" in existing_features
    ):
        requested.update(
            {
                "Time",
                "Point_in_Time",
                "Duration",
                "State",
                "Communication",
                "Statement",
                "Question",
            }
        )

    return [
        feature
        for feature in LLM_DERIVED_FEATURES
        if (
            feature in requested
            and feature not in existing_features
        )
    ]


def classify_common_noun_features(
    token: Any,
    selected_synset: Any,
    sentence: Any,
    sentence_index: int,
    sentences: Sequence[Any],
    hypernym_paths: Sequence[Sequence[Any]],
    existing_features: Set[str],
) -> Tuple[Set[str], Dict[str, List[str]], Dict[str, Any]]:
    """Ask about features not reliably recoverable from WordNet."""

    requested_features = choose_llm_noun_features(
        existing_features
    )

    if not requested_features:
        return set(), {}, {"features": {}}

    payload = {
        "target": {
            "form": token.get("form"),
            "lemma": token.get("lemma"),
            "upos": token.get("upos"),
        },
        "selected_wordnet_sense": synset_as_candidate(
            selected_synset
        ),
        "context": build_context(
            sentences,
            sentence_index,
        ),
        "syntax": token_dependency_context(
            sentence,
            token,
        ),
        "wordnet_hypernym_paths": [
            [
                synset.name()
                for synset in path
            ]
            for path in hypernym_paths
        ],
        "already_established_features": sorted(
            existing_features
        ),
        "requested_features": requested_features,
    }

    response = call_llm_json(
        task="common_noun_semantic_features_v2",
        system_prompt=NOUN_FEATURE_SYSTEM_PROMPT,
        user_payload=payload,
    )

    response_features = response.get("features", {})
    accepted: Set[str] = set()
    sources: Dict[str, List[str]] = {}

    if not isinstance(response_features, dict):
        response_features = {}

    for feature in requested_features:
        decision = response_features.get(feature, {})

        if not isinstance(decision, dict):
            continue

        if normalise_boolean(decision.get("applies")):
            accepted.add(feature)

            reason = str(
                decision.get("reason", "")
            ).strip()

            sources[feature] = [
                f"llm:{reason}" if reason else "llm"
            ]

    if "Rigid" in accepted and "Non_Rigid" in accepted:
        accepted.discard("Rigid")
        accepted.discard("Non_Rigid")
        sources.pop("Rigid", None)
        sources.pop("Non_Rigid", None)

    if (
        "Point_in_Time" in accepted
        and "Duration" in accepted
    ):
        accepted.discard("Point_in_Time")
        accepted.discard("Duration")

        sources.pop("Point_in_Time", None)
        sources.pop("Duration", None)


        accepted.add("Time")
        sources.setdefault(
            "Time",
            ["llm:ambiguous temporal construal"],
        )

    if (
        "Abstract" in accepted
    ):
        accepted.discard("Inanimate")
        sources.pop("Inanimate", None)

    if "Solid" not in accepted:
        for feature in {
            "Rigid",
            "Non_Rigid",
            "Elongated",
            "Pointy",
        }:
            accepted.discard(feature)
            sources.pop(feature, None)

    return accepted, sources, response


# =============================================================================
# Proper-noun feature classification
# =============================================================================

PROPER_NOUN_SYSTEM_PROMPT = """
You are classifying the contextual referent of a proper noun.

Classify only the entity denoted by this occurrence in the supplied sentence.
Do not classify other entities that can share the same name.

Available features:

Organisation:
    A company, institution, government, government body, association, team, agency, or
    other organised collective.

Human:
    One individual human being, including a fictional human character.

Masculine:
    A human that is masculine in gender.

Feminine:
    A human that is feminine in gender.

Unspecified_Gender_or_Neuter:
    A human referent that is unspecified for gender or gender-neutral.

Animal:
    A non-human animal, including an individually named animal.

Vehicle:
    A particular car, ship, aircraft, train, spacecraft, or other vehicle.

Language:
    A named natural language or named linguistic variety.

Eventive:
    A named event, occurrence, war, festival, meeting, election, competition, disaster, performance, or similar happening. A diagnostic is whether the word can logically and grammatically be paired with "during". E.g. "during the party", "during the war".

Location:
    A named place, country, region, settlement, building used as a location,
    geographical feature, site, or spatial area.

Web_Location:
    A location on the internet.

Internal_Control:
    The referent can characteristically initiate, regulate, or direct its own
    actions or behaviour. Humans, many animals, organisations acting through
    members, and autonomous machines may qualify. Natural forces such as rain and wind also qualify.
    Mere movement or causation is insufficient.

Time:
    The referent is a temporal entity, interval, date, age, period, or temporal position. The referent should be able to logically answer questions beginning with "when" or "how long" or "for how long".
    E.g. "for four hours", "next week", "in a day", "tomorrow", "on the 6th of July", "in July", "in 2 hours", "at 4:00 pm", "in three minutes".

Point_in_Time:
    The temporal referent is conceptualised as a bounded point or date rather than an extended interval. The referent should be able to logically answer questions beginning with "when".
    E.g. "at 4:00 pm", "tomorrow", "yesterday", "in July", "in 1980", "at the weekend", "on the weekend".
    Point_in_Time and Duration should not both be true unless the sentence genuinely coerces both readings.

Duration:
    The temporal referent is conceptualised as an interval or amount of time. The referent should be able to logically answer questions beginning with "how long" or "for how long".
    E.g. "I ran for four hours", "I ran to the next city in a day", "I trained hard for all of December", "I trained hard throughout the week".
    Point_in_Time and Duration should not both be true unless the sentence genuinely coerces both readings.

Rules:

1. Several features may be true only when the contextual referent genuinely
   satisfies all of them.
2. A country may be Location, Organisation, or both, depending on whether the
   sentence refers to its territory or its government or institution.
3. A building name is Location when it refers to the site, but Organisation
   when it metonymically denotes the institution.
4. A sports-team name is Organisation, not Human.
5. A named event is Eventive, not Location merely because it happens
   somewhere.
6. Return a decision for every supplied feature.
7. Be conservative and give a brief contextual justification.

Return exactly:

{
  "features": {
    "Organisation": {
      "applies": false,
      "reason": "brief justification"
    },
    "Human": {
      "applies": false,
      "reason": "brief justification"
    }
  }
}

Include every requested feature in the features object.
""".strip()


def classify_proper_noun(
    token: Any,
    sentence: Any,
    sentence_index: int,
    sentences: Sequence[Any],
) -> Tuple[Set[str], Dict[str, List[str]], Dict[str, Any]]:
    """Classify a PROPN without forcing it into WordNet."""

    payload = {
        "target": {
            "form": token.get("form"),
            "lemma": token.get("lemma"),
            "upos": token.get("upos"),
        },
        "context": build_context(
            sentences,
            sentence_index,
        ),
        "syntax": token_dependency_context(
            sentence,
            token,
        ),
        "requested_features": list(
            PROPER_NOUN_FEATURES
        ),
    }

    response = call_llm_json(
        task="proper_noun_referent_features_v2",
        system_prompt=PROPER_NOUN_SYSTEM_PROMPT,
        user_payload=payload,
    )

    response_features = response.get("features", {})
    accepted: Set[str] = set()
    sources: Dict[str, List[str]] = {}

    if not isinstance(response_features, dict):
        response_features = {}

    for feature in PROPER_NOUN_FEATURES:
        decision = response_features.get(feature, {})

        if not isinstance(decision, dict):
            continue

        if normalise_boolean(decision.get("applies")):
            accepted.add(feature)

            reason = str(
                decision.get("reason", "")
            ).strip()

            sources[feature] = [
                f"llm:{reason}" if reason else "llm"
            ]

    return accepted, sources, response


# =============================================================================
# Output helpers
# =============================================================================

def append_jsonl(
    path: Path,
    record: Dict[str, Any],
) -> None:
    """Append one record and force it to disk immediately."""

    with path.open("a", encoding="utf-8") as handle:
        handle.write(
            json.dumps(
                record,
                ensure_ascii=False,
            )
            + "\n"
        )

        handle.flush()
        os.fsync(handle.fileno())


def merge_feature_sources(
    destination: Dict[str, List[str]],
    additions: Dict[str, List[str]],
) -> None:
    """Merge feature provenance without duplicates."""

    for feature, source_values in additions.items():
        destination.setdefault(feature, [])

        for source in source_values:
            if source not in destination[feature]:
                destination[feature].append(source)


def add_inherited_feature_sources(
    direct_features: Set[str],
    all_features: Set[str],
    sources: Dict[str, List[str]],
) -> None:
    """Mark features introduced only by inheritance."""

    for feature in all_features:
        if feature not in direct_features:
            sources.setdefault(
                feature,
                ["inherited"],
            )


# =============================================================================
# Token processing
# =============================================================================

def process_noun_or_verb(
    token: Any,
    sentence: Any,
    sentence_index: int,
    sentences: Sequence[Any],
    prior_wsd_entries: Sequence[Dict[str, Any]],
) -> Dict[str, Any]:
    """Perform WSD and, for nouns, feature analysis."""

    form = str(token.get("form") or "")
    lemma = token.get("lemma")
    upos = str(token.get("upos") or "")

    candidates = get_wordnet_candidates(
        form=form,
        lemma=str(lemma) if lemma else None,
        upos=upos,
    )

    base_record: Dict[str, Any] = {
        "sentence_index": sentence_index,
        "sentence_id": sentence.metadata.get("sent_id"),
        "sentence_text": sentence_text(sentence),
        "token_id": token.get("id"),
        "form": form,
        "lemma": lemma,
        "upos": upos,
        "deprel": token.get("deprel"),
        "head": token.get("head"),
        "candidate_synsets": [
            synset.name()
            for synset in candidates
        ],
        "selected_synset": None,
        "selected_definition": None,
        "wsd_choice": None,
        "wsd_method": None,
        "wsd_raw_response": None,
        "hypernym_paths": [],
        "features": [],
        "feature_sources": {},
        "status": None,
    }

    if not candidates:
        base_record["status"] = "no_wordnet_candidates"
        return base_record

    selected_synset, wsd_response = disambiguate_token(
        sentence=sentence,
        sentence_index=sentence_index,
        sentences=sentences,
        token=token,
        candidates=candidates,
        prior_wsd_entries=prior_wsd_entries,
    )

    base_record["wsd_choice"] = wsd_response.get("choice")
    base_record["wsd_method"] = wsd_response.get("method")
    base_record["wsd_raw_response"] = wsd_response.get(
        "raw_response"
    )

    if selected_synset is None:
        base_record["status"] = "no_valid_sense_selected"
        base_record["wsd_raw"] = wsd_response
        return base_record

    base_record["selected_synset"] = selected_synset.name()
    base_record["selected_definition"] = (
        selected_synset.definition()
    )

    if upos == "VERB":
        base_record["status"] = "wsd_complete"
        return base_record

    hypernym_paths = get_noun_hypernym_paths(
        selected_synset
    )

    base_record["hypernym_paths"] = serialise_hypernym_paths(
        hypernym_paths
    )

    wordnet_features, feature_sources = (
        features_from_hypernym_paths(
            hypernym_paths
        )
    )

    direct_feature_snapshot = set(wordnet_features)

    llm_features, llm_sources, llm_response = (
        classify_common_noun_features(
            token=token,
            selected_synset=selected_synset,
            sentence=sentence,
            sentence_index=sentence_index,
            sentences=sentences,
            hypernym_paths=hypernym_paths,
            existing_features=wordnet_features,
        )
    )

    combined_direct_features = (
        wordnet_features.union(llm_features)
    )

    combined_features = expand_feature_inheritance(
        combined_direct_features
    )

    merge_feature_sources(
        feature_sources,
        llm_sources,
    )

    add_inherited_feature_sources(
        direct_features=(
            direct_feature_snapshot.union(llm_features)
        ),
        all_features=combined_features,
        sources=feature_sources,
    )

    base_record["features"] = sorted(
        combined_features
    )

    base_record["feature_sources"] = {
        feature: feature_sources.get(
            feature,
            ["inherited"],
        )
        for feature in sorted(combined_features)
    }

    base_record["llm_feature_response"] = llm_response
    base_record["status"] = "wsd_and_features_complete"

    return base_record


def process_proper_noun(
    token: Any,
    sentence: Any,
    sentence_index: int,
    sentences: Sequence[Any],
) -> Dict[str, Any]:
    """Classify a proper noun's contextual referent."""

    llm_features, llm_sources, response = (
        classify_proper_noun(
            token=token,
            sentence=sentence,
            sentence_index=sentence_index,
            sentences=sentences,
        )
    )

    direct_features = set(llm_features)

    all_features = expand_feature_inheritance(
        direct_features
    )

    add_inherited_feature_sources(
        direct_features=direct_features,
        all_features=all_features,
        sources=llm_sources,
    )

    return {
        "sentence_index": sentence_index,
        "sentence_id": sentence.metadata.get("sent_id"),
        "sentence_text": sentence_text(sentence),
        "token_id": token.get("id"),
        "form": token.get("form"),
        "lemma": token.get("lemma"),
        "upos": token.get("upos"),
        "deprel": token.get("deprel"),
        "head": token.get("head"),
        "candidate_synsets": [],
        "selected_synset": None,
        "selected_definition": None,
        "wsd_choice": None,
        "wsd_method": None,
        "wsd_raw_response": None,
        "hypernym_paths": [],
        "features": sorted(all_features),
        "feature_sources": {
            feature: llm_sources.get(
                feature,
                ["inherited"],
            )
            for feature in sorted(all_features)
        },
        "proper_noun_feature_response": response,
        "status": "proper_noun_features_complete",
    }


# =============================================================================
# Console output
# =============================================================================

def print_result(result: Dict[str, Any]) -> None:
    """Print the selected sense and selected features."""

    status = result.get("status")
    selected_synset = result.get("selected_synset")
    selected_definition = result.get("selected_definition")
    selected_features = result.get("features", [])
    feature_sources = result.get("feature_sources", {})
    wsd_method = result.get("wsd_method")

    print(f"  Status: {status}")

    if wsd_method:
        print(f"  WSD method: {wsd_method}")

    if selected_synset:
        print(f"  Selected sense: {selected_synset}")

        if selected_definition:
            print(
                f"  Definition: {selected_definition}"
            )

    if selected_features:
        print("  Selected features:")

        for feature in selected_features:
            sources = feature_sources.get(feature, [])
            source_text = (
                "; ".join(sources)
                if sources
                else "unknown"
            )

            print(
                f"    - {feature}: {source_text}"
            )
    else:
        print("  Selected features: none")

    if status == "error":
        print(
            "  Error: "
            f"{result.get('error_type')}: "
            f"{result.get('error')}"
        )

    print()



def print_sentence_progress(sentence, current_token_id, history_by_sentence, sentence_index):
    history = {
        h["token_id"]: h
        for h in history_by_sentence.get(sentence_index, [])
    }

    parts = []

    for token in sentence:
        token_id = token.get("id")

        if not isinstance(token_id, int):
            continue

        text = token["form"]

        if token_id == current_token_id:
            parts.append(f">>{text}<<")

        elif token_id in history:
            syn = history[token_id]["synset"]
            parts.append(f"{text}{{{syn}}}")

        else:
            parts.append(text)

    print("\nSentence:")
    print(" ".join(parts))
    print()


# =============================================================================
# Main execution
# =============================================================================

def main() -> None:
    print(
        "Loading CoNLL-U data from:\n"
        f"  {CONLLU_PATH.resolve()}"
    )

    sentences = load_conllu(CONLLU_PATH)

    print(f"Loaded {len(sentences):,} sentences.")

    completed, history_by_sentence = load_processing_state(
        OUTPUT_PATH
    )

    if completed:
        print(
            f"Restart mode: found {len(completed):,} completed "
            f"token records in {OUTPUT_PATH}."
        )

    restored_wsd_count = sum(
        len(entries)
        for entries in history_by_sentence.values()
    )

    if restored_wsd_count:
        print(
            f"Restored {restored_wsd_count:,} successful earlier "
            "WSD decisions."
        )

    processed_this_run = 0

    try:
        for sentence_index, sentence in enumerate(sentences):
            print(f"[sentence {sentence_index+1}/{len(sentences)}; ")
            for token in sentence:
                token_id = token.get("id")

                # Skip multiword-token ranges and empty nodes.
                if not isinstance(token_id, int):
                    continue

                upos = token.get("upos")

                print(f"token {token_id}] {token['form']!r} ({upos})")
                
                print_sentence_progress(
                    sentence=sentence,
                    current_token_id=token_id,
                    history_by_sentence=history_by_sentence,
                    sentence_index=sentence_index,
                )

                if upos not in {
                    "NOUN",
                    "VERB",
                    "PROPN",
                }:
                    continue

                completion_key = (
                    sentence_index,
                    str(token_id),
                )

                if completion_key in completed:
                    continue

                print(
                    f"[sentence "
                    f"{sentence_index + 1}/{len(sentences)}; "
                    f"token {token_id}] "
                    f"{token.get('form')!r} ({upos})"
                )

                try:
                    if upos == "PROPN":
                        result = process_proper_noun(
                            token=token,
                            sentence=sentence,
                            sentence_index=sentence_index,
                            sentences=sentences,
                        )

                    else:
                        prior_wsd_entries = (
                            get_prior_wsd_entries(
                                history_by_sentence=(
                                    history_by_sentence
                                ),
                                sentence_index=sentence_index,
                                current_token_id=token_id,
                            )
                        )

                        result = process_noun_or_verb(
                            token=token,
                            sentence=sentence,
                            sentence_index=sentence_index,
                            sentences=sentences,
                            prior_wsd_entries=(
                                prior_wsd_entries
                            ),
                        )

                except Exception as exc:
                    result = {
                        "sentence_index": sentence_index,
                        "sentence_id": sentence.metadata.get(
                            "sent_id"
                        ),
                        "sentence_text": sentence_text(sentence),
                        "token_id": token_id,
                        "form": token.get("form"),
                        "lemma": token.get("lemma"),
                        "upos": upos,
                        "features": [],
                        "feature_sources": {},
                        "status": "error",
                        "error_type": type(exc).__name__,
                        "error": str(exc),
                    }

                append_jsonl(
                    OUTPUT_PATH,
                    result,
                )

                completed.add(completion_key)
                processed_this_run += 1

                selected_synset = result.get(
                    "selected_synset"
                )

                selected_definition = result.get(
                    "selected_definition"
                )

                if (
                    upos in {"NOUN", "VERB"}
                    and selected_synset
                ):
                    add_wsd_history_entry(
                        history_by_sentence=(
                            history_by_sentence
                        ),
                        sentence_index=sentence_index,
                        token_id=token_id,
                        form=str(
                            token.get("form") or ""
                        ),
                        lemma=token.get("lemma"),
                        upos=str(upos),
                        synset=str(selected_synset),
                        definition=str(
                            selected_definition or ""
                        ),
                    )

                print_result(result)

    finally:
        save_llm_cache(force=True)

    print()
    print(
        f"Processed {processed_this_run:,} new token records."
    )
    print(f"Results: {OUTPUT_PATH.resolve()}")
    print(f"LLM cache: {CACHE_PATH.resolve()}")


if __name__ == "__main__":
    main()

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


Loading CoNLL-U data from:
  /home/mark/verbnet-roberta/UD_English-EWT-master/UD_English-EWT-master/en_ewt-ud-train.conllu
Loaded 7,000 sentences.
Restart mode: found 20,070 completed token records in ewt_wordnet_wsd_features.jsonl.
Restored 8,335 successful earlier WSD decisions.
[sentence 1/7000; 
token 1] 'Al' (PROPN)

Sentence:
>>Al<< - Zaman : American forces{military_unit.n.01} killed Shaikh Abdullah al - Ani , the preacher{preacher.n.01} at the mosque{mosque.n.01} in the town{town.n.01} of Qaim , near the Syrian border{boundary_line.n.01} .

token 2] '-' (PUNCT)

Sentence:
Al >>-<< Zaman : American forces{military_unit.n.01} killed Shaikh Abdullah al - Ani , the preacher{preacher.n.01} at the mosque{mosque.n.01} in the town{town.n.01} of Qaim , near the Syrian border{boundary_line.n.01} .

token 3] 'Zaman' (PROPN)

Sentence:
Al - >>Zaman<< : American forces{military_unit.n.01} killed Shaikh Abdullah al - Ani , the preacher{preacher.n.01} at the mosque{mosque.n.01} in the town{to

IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)



  Status: no_valid_sense_selected
  WSD method: llm_numbered_choice
  Selected features: none

token 31] 'in' (ADP)

Sentence:
If this is the case{event.n.02} I would like someone to walk me through June to see if the numbers{number.n.01} did really work themselves out as there was a large swing >>in<< second order on June 29 that was not adjusted for .

token 32] 'second' (ADJ)

Sentence:
If this is the case{event.n.02} I would like someone to walk me through June to see if the numbers{number.n.01} did really work themselves out as there was a large swing in >>second<< order on June 29 that was not adjusted for .

token 33] 'order' (NOUN)

Sentence:
If this is the case{event.n.02} I would like someone to walk me through June to see if the numbers{number.n.01} did really work themselves out as there was a large swing in second >>order<< on June 29 that was not adjusted for .

[sentence 3374/7000; token 33] 'order' (NOUN)
WSD request failed on attempt 1: The WSD response did not contain